In [1]:
# =========================
# GoEmotions EXP1: DistilBERT Baseline
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)
print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")
print(f"Num labels: {NUM_LABELS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GoEmotionsDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        enc = self.tokenizer(text, max_length=self.max_len, truncation=True,
                             padding="max_length", return_tensors="pt")
        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
        }

class DistilBertBaseline(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # first token
        return self.classifier(cls)

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].cpu().numpy()
            logits = model(ids, mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs); all_labels.append(y)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)

    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)

    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsDataset(ds["train"], tokenizer, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsDataset(ds["validation"], tokenizer, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsDataset(ds["test"], tokenizer, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

model = DistilBertBaseline(MODEL_NAME, NUM_LABELS).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "distilbert_exp1_best.pt"

print("\nTraining EXP1...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


/home/jovyan/hf-goemotions/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
Num labels: 28


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 400a508e-de51-44a4-b3fa-5f7c42888bf6)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



Training EXP1...
Epoch 1/4 | loss=0.1332 | val_macro=0.3770 | val_micro=0.5325
  ✓ saved best (val_macro=0.3770)
Epoch 2/4 | loss=0.0785 | val_macro=0.4387 | val_micro=0.5700
  ✓ saved best (val_macro=0.4387)
Epoch 3/4 | loss=0.0615 | val_macro=0.4861 | val_micro=0.5798
  ✓ saved best (val_macro=0.4861)
Epoch 4/4 | loss=0.0459 | val_macro=0.4911 | val_micro=0.5806
  ✓ saved best (val_macro=0.4911)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.934718,0.894886,0.914369,352,337,352.0,0.0621
1,amusement,0.793594,0.844697,0.818349,264,281,264.0,0.0518
2,love,0.789256,0.802521,0.795833,238,242,238.0,0.0446
3,admiration,0.689861,0.688492,0.689176,504,503,504.0,0.0927
4,fear,0.653846,0.653846,0.653846,78,78,78.0,0.0144
5,neutral,0.679743,0.591494,0.632555,1787,1555,1787.0,0.2865
6,joy,0.654930,0.577640,0.613861,161,142,161.0,0.0262
7,remorse,0.545455,0.642857,0.590164,56,66,56.0,0.0122
8,sadness,0.638462,0.532051,0.580420,156,130,156.0,0.0240
9,optimism,0.618421,0.505376,0.556213,186,152,186.0,0.0280


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,caring,0.505263,0.355556,0.417391,135,95,135.0,0.0175
19,disapproval,0.436019,0.344569,0.384937,267,211,267.0,0.0389
20,pride,0.800000,0.250000,0.380952,16,5,16.0,0.0009
21,approval,0.460581,0.316239,0.375000,351,241,351.0,0.0444
22,annoyance,0.475410,0.271875,0.345924,320,183,320.0,0.0337
23,disappointment,0.457143,0.211921,0.289593,151,70,151.0,0.0129
24,nervousness,0.357143,0.217391,0.270270,23,14,23.0,0.0026
25,realization,0.416667,0.172414,0.243902,145,60,145.0,0.0111
26,relief,1.000000,0.090909,0.166667,11,1,11.0,0.0002
27,grief,0.000000,0.000000,0.000000,6,0,6.0,0.0000



TEST @0.5 | Macro-F1=0.4841 | Micro-F1=0.5838
Done.


In [2]:
# =========================
# GoEmotions EXP2: DistilBERT + pos_weight (class weights)
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)
print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GoEmotionsDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        enc = self.tokenizer(text, max_length=self.max_len, truncation=True,
                             padding="max_length", return_tensors="pt")
        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
        }

class DistilBertBaseline(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

def compute_pos_weights(hf_train_split, num_labels, device):
    pos = np.zeros(num_labels, dtype=np.float64)
    for ex in hf_train_split:
        for lab in ex["labels"]:
            pos[lab] += 1.0
    N = len(hf_train_split)
    neg = N - pos
    pos = np.clip(pos, 1.0, None)
    pos_w = neg / pos
    pos_w = np.clip(pos_w, 1.0, 50.0)
    return torch.tensor(pos_w, dtype=torch.float32, device=device)

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].cpu().numpy()
            logits = model(ids, mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs); all_labels.append(y)
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)
    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)
    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsDataset(ds["train"], tokenizer, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsDataset(ds["validation"], tokenizer, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsDataset(ds["test"], tokenizer, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

model = DistilBertBaseline(MODEL_NAME, NUM_LABELS).to(DEVICE)

pos_wts = compute_pos_weights(ds["train"], NUM_LABELS, DEVICE)
print(f"[EXP2] pos_weight range: {pos_wts.min().item():.2f} - {pos_wts.max().item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_wts)
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "distilbert_exp2_posweight_best.pt"

print("\nTraining EXP2...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
[EXP2] pos_weight range: 2.05 - 50.00

Training EXP2...
Epoch 1/4 | loss=0.6838 | val_macro=0.3983 | val_micro=0.4032
  ✓ saved best (val_macro=0.3983)
Epoch 2/4 | loss=0.4485 | val_macro=0.4099 | val_micro=0.4400
  ✓ saved best (val_macro=0.4099)
Epoch 3/4 | loss=0.3137 | val_macro=0.4452 | val_micro=0.4849
  ✓ saved best (val_macro=0.4452)
Epoch 4/4 | loss=0.2205 | val_macro=0.4621 | val_micro=0.5116
  ✓ saved best (val_macro=0.4621)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.740659,0.957386,0.835192,352,455,352.0,0.0838
1,amusement,0.681199,0.946970,0.792393,264,367,264.0,0.0676
2,love,0.644118,0.920168,0.757785,238,340,238.0,0.0626
3,neutral,0.626074,0.733632,0.675599,1787,2094,1787.0,0.3858
4,admiration,0.531095,0.847222,0.652905,504,804,504.0,0.1481
5,remorse,0.481481,0.928571,0.634146,56,108,56.0,0.0199
6,fear,0.427632,0.833333,0.565217,78,152,78.0,0.0280
7,curiosity,0.395100,0.908451,0.550694,284,653,284.0,0.1203
8,surprise,0.381481,0.730496,0.501217,141,270,141.0,0.0498
9,sadness,0.329577,0.750000,0.457926,156,355,156.0,0.0654


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,embarrassment,0.253521,0.486486,0.333333,37,71,37.0,0.0131
19,confusion,0.204969,0.862745,0.331242,153,644,153.0,0.1187
20,pride,0.259259,0.437500,0.325581,16,27,16.0,0.0050
21,annoyance,0.213115,0.650000,0.320988,320,976,320.0,0.1798
22,approval,0.212121,0.638177,0.318408,351,1056,351.0,0.1946
23,excitement,0.205521,0.650485,0.312354,103,326,103.0,0.0601
24,relief,0.170213,0.727273,0.275862,11,47,11.0,0.0087
25,nervousness,0.175676,0.565217,0.268041,23,74,23.0,0.0136
26,disappointment,0.157667,0.483444,0.237785,151,463,151.0,0.0853
27,realization,0.137860,0.462069,0.212361,145,486,145.0,0.0896



TEST @0.5 | Macro-F1=0.4507 | Micro-F1=0.5022
Done.


In [3]:
# =========================
# GoEmotions EXP3: DistilBERT + Emoji Priors (learned from TRAIN)
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

# emoji lib
try:
    import emoji
except Exception:
    import sys
    !{sys.executable} -m pip -q install emoji
    import emoji

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

EMOJI_MIN_COUNT = 10
EMOJI_MAX = 300
EMOJI_PRIOR_SCALE = 0.35

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)
print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def extract_emojis_from_text(text: str):
    return [ch for ch in text if ch in emoji.EMOJI_DATA]

def build_emoji_priors_from_train(train_split, num_labels=28, min_count=10, max_emojis=300):
    emoji_counts = {}
    emoji_label_counts = {}
    for ex in train_split:
        text = ex["text"]
        labs = ex["labels"]
        ems = extract_emojis_from_text(text)
        if not ems:
            continue
        for em in ems:
            emoji_counts[em] = emoji_counts.get(em, 0) + 1
            if em not in emoji_label_counts:
                emoji_label_counts[em] = np.zeros(num_labels, dtype=np.float64)
            for lab in labs:
                emoji_label_counts[em][lab] += 1.0

    filtered = [(em, c) for em, c in emoji_counts.items() if c >= min_count]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_emojis]
    if len(filtered) == 0:
        print("⚠️ No frequent emojis found. Disabling emoji priors.")
        return [], None

    emoji_list = [em for em, _ in filtered]
    prior = torch.zeros(len(emoji_list), num_labels, dtype=torch.float32)

    for i, em in enumerate(emoji_list):
        counts = emoji_label_counts.get(em, np.zeros(num_labels, dtype=np.float64))
        sm = counts + 1.0
        sm = sm / sm.sum()
        prior[i] = torch.tensor(sm, dtype=torch.float32)

    print(f"✓ Learned emoji priors from TRAIN: {len(emoji_list)} emojis (min_count={min_count}, max={max_emojis})")
    return emoji_list, prior

emoji_list, emoji_prior = build_emoji_priors_from_train(
    ds["train"], num_labels=NUM_LABELS, min_count=EMOJI_MIN_COUNT, max_emojis=EMOJI_MAX
)

class GoEmotionsEmojiDataset(Dataset):
    def __init__(self, hf_split, tokenizer, emoji_list, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels
        self.emoji_list = emoji_list
        self.emoji_to_idx = {e:i for i,e in enumerate(emoji_list)}

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        enc = self.tokenizer(text, max_length=self.max_len, truncation=True,
                             padding="max_length", return_tensors="pt")

        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0

        evec = torch.zeros(len(self.emoji_list), dtype=torch.float32)
        if self.emoji_list:
            for ch in extract_emojis_from_text(text):
                j = self.emoji_to_idx.get(ch, None)
                if j is not None:
                    evec[j] = 1.0

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
            "emoji_vec": evec,
        }

class DistilBertWithEmojiPriors(nn.Module):
    def __init__(self, model_name, num_labels, emoji_prior_matrix=None, prior_scale=0.35):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)

        self.prior_scale = prior_scale
        if emoji_prior_matrix is not None:
            self.register_buffer("emoji_prior", emoji_prior_matrix)  # [E, L]
        else:
            self.emoji_prior = None

    def forward(self, input_ids, attention_mask, emoji_vec=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)

        if (self.emoji_prior is not None) and (emoji_vec is not None) and (emoji_vec.size(1) == self.emoji_prior.size(0)):
            bias_raw = emoji_vec @ self.emoji_prior
            denom = emoji_vec.sum(dim=1, keepdim=True).clamp(min=1.0)
            bias = bias_raw / denom
            logits = logits + (self.prior_scale * bias)

        return logits

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            evec = batch["emoji_vec"].to(device)
            y = batch["labels"].cpu().numpy()

            logits = model(ids, mask, emoji_vec=evec)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs); all_labels.append(y)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)

    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)

    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsEmojiDataset(ds["train"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsEmojiDataset(ds["validation"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsEmojiDataset(ds["test"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

model = DistilBertWithEmojiPriors(
    MODEL_NAME, NUM_LABELS,
    emoji_prior_matrix=emoji_prior,
    prior_scale=EMOJI_PRIOR_SCALE
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "distilbert_exp3_emoji_priors_best.pt"

print("\nTraining EXP3...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        evec = batch["emoji_vec"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask, emoji_vec=evec)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
✓ Learned emoji priors from TRAIN: 36 emojis (min_count=10, max=300)

Training EXP3...
Epoch 1/4 | loss=0.1332 | val_macro=0.3798 | val_micro=0.5338
  ✓ saved best (val_macro=0.3798)
Epoch 2/4 | loss=0.0785 | val_macro=0.4415 | val_micro=0.5708
  ✓ saved best (val_macro=0.4415)
Epoch 3/4 | loss=0.0615 | val_macro=0.4879 | val_micro=0.5798
  ✓ saved best (val_macro=0.4879)
Epoch 4/4 | loss=0.0459 | val_macro=0.4925 | val_micro=0.5817
  ✓ saved best (val_macro=0.4925)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.942771,0.889205,0.915205,352,332,352.0,0.0612
1,amusement,0.793594,0.844697,0.818349,264,281,264.0,0.0518
2,love,0.777328,0.806723,0.791753,238,247,238.0,0.0455
3,admiration,0.686508,0.686508,0.686508,504,504,504.0,0.0929
4,fear,0.653846,0.653846,0.653846,78,78,78.0,0.0144
5,neutral,0.681935,0.591494,0.633503,1787,1550,1787.0,0.2856
6,joy,0.659864,0.602484,0.629870,161,147,161.0,0.0271
7,remorse,0.569231,0.660714,0.611570,56,65,56.0,0.0120
8,sadness,0.643411,0.532051,0.582456,156,129,156.0,0.0238
9,optimism,0.625850,0.494624,0.552553,186,147,186.0,0.0271


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,caring,0.500000,0.362963,0.420601,135,98,135.0,0.0181
19,approval,0.453488,0.333333,0.384236,351,258,351.0,0.0475
20,pride,0.800000,0.250000,0.380952,16,5,16.0,0.0009
21,disapproval,0.418605,0.337079,0.373444,267,215,267.0,0.0396
22,annoyance,0.486188,0.275000,0.351297,320,181,320.0,0.0334
23,disappointment,0.476190,0.198675,0.280374,151,63,151.0,0.0116
24,nervousness,0.384615,0.217391,0.277778,23,13,23.0,0.0024
25,realization,0.393443,0.165517,0.233010,145,61,145.0,0.0112
26,relief,1.000000,0.090909,0.166667,11,1,11.0,0.0002
27,grief,0.000000,0.000000,0.000000,6,0,6.0,0.0000



TEST @0.5 | Macro-F1=0.4863 | Micro-F1=0.5842
Done.
